# Point Source to Far Field Refractor — Benchmark

Implements the refraction example from **Figure 4** of the paper:
κ = 0.6, cost c(x,y) = −log(1 − 0.6·(x·y)),
source Ω = {θ ∈ [π/12, π/3], φ ∈ [π/12, π/4]},
target Ω\* = {θ ∈ [π/10, π/5], φ ∈ [π/10, π/5]}.
Both patches live on the **upper hemisphere**.

**Kernel → Restart & Run All** runs the full pipeline end-to-end:

| Step | What happens |
|------|--------------|
| 1 | **Config** — sizes, benchmark case, patch bounds, κ |
| 2 | **Dependencies** — ensure numpy, matplotlib, plotly are installed |
| 3 | **Generate** quasi-random point clouds on the sphere (refraction patches) |
| 4 | **Compile** C++ via `make` (auto-selects Snell's law via `#ifdef`) |
| 5 | **Run** the benchmark |
| 6 | **Find output** + define data-loading helpers |
| 7 | **Visualize** (matplotlib, 6-panel static PNG) |
| 8 | **Interactive 3D** — point-cloud view + ray diagram (Plotly) |

## Step 1 — Configuration

In [ ]:
import os, sys

# ── Point-cloud sizes ─────────────────────────────────────────────────────────
# NK complexity is O(NK²):  1600 → fast (seconds)  |  16488 → full (minutes)
NK       = 1600
NK_small = 200    # warm-start sampler (must be < NK)

# ── Benchmark test-case ───────────────────────────────────────────────────────
BENCHMARK = 'test_3D_FlatPatchRefraction_logcost_MonteCarlo.h'

# ── Refraction parameter ──────────────────────────────────────────────────────
# Must match COST_K in Generic_3D_refractioncost_MonteCarlo.h (= 0.6)
COST_K = 0.6

# ── Patch bounds (multiples of π) ─────────────────────────────────────────────
# Paper Figure 4: source θ∈[π/12,π/3], φ∈[π/12,π/4];
#                 target θ∈[π/10,π/5], φ∈[π/10,π/5].
# IMPORTANT: both patches are on the UPPER hemisphere (θ < π/2, i.e. z > 0).
SRC_THETA_MIN = 1/12   # 15°
SRC_THETA_MAX = 1/3    # 60°
SRC_PHI_MIN   = 1/12   # 15°
SRC_PHI_MAX   = 1/4    # 45°

TGT_THETA_MIN = 1/10   # 18°
TGT_THETA_MAX = 1/5    # 36°
TGT_PHI_MIN   = 1/10   # 18°
TGT_PHI_MAX   = 1/5    # 36°

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = os.path.abspath('')
CODE_DIR  = os.path.join(REPO_ROOT, 'BenchmarkCode')

print(f'Python    : {sys.executable}')
print(f'Repo root : {REPO_ROOT}')
print(f'Code dir  : {CODE_DIR}')
print(f'NK={NK}, NK_small={NK_small}, κ={COST_K}')
print(f'Benchmark : {BENCHMARK}')
print(f'Source patch  θ: [{SRC_THETA_MIN:.4f}π, {SRC_THETA_MAX:.4f}π]  φ: [{SRC_PHI_MIN:.4f}π, {SRC_PHI_MAX:.4f}π]')
print(f'Target patch  θ: [{TGT_THETA_MIN:.4f}π, {TGT_THETA_MAX:.4f}π]  φ: [{TGT_PHI_MIN:.4f}π, {TGT_PHI_MAX:.4f}π]')

## Step 2 — Install dependencies

Installs numpy, matplotlib, and plotly into **this kernel** via `sys.executable`.

In [ ]:
import subprocess

def pip(*args):
    cmd = [sys.executable, '-m', 'pip', *args]
    r = subprocess.run(cmd, capture_output=True, text=True)
    for line in (r.stdout + r.stderr).splitlines():
        if any(k in line for k in ('ERROR', 'error', 'WARNING', 'Successfully', 'already')):
            print(line)
    return r.returncode

print('Installing numpy ...')
pip('install', 'numpy>=1.21,<2.0', '--upgrade', '--quiet')

print('Installing matplotlib ...')
pip('install', 'matplotlib', '--upgrade', '--force-reinstall', '--quiet')

print('Installing plotly ...')
pip('install', 'plotly', '--upgrade', '--quiet')

r = subprocess.run(
    [sys.executable, '-c',
     'import numpy, matplotlib, plotly; '
     'print("numpy", numpy.__version__, "| matplotlib", matplotlib.__version__, "| plotly", plotly.__version__)'],
    capture_output=True, text=True
)
print(r.stdout.strip() or r.stderr.strip())
print('✓ Dependencies ready')

## Step 3 — Generate point clouds

Calls `BenchmarkCode/generate_pointclouds.py` with **refraction patch bounds**
(both source and target on the upper hemisphere).
Writes three header files:

| File | Contents |
|------|----------|
| `QuasiMonteCarlo/MonteCarlo_Pointcloud_3D_128.h` | NK source + target points |
| `SmallGrid/3D_MonteCarlo_Pointcloud_small.h` | NK_small warm-start points |
| `PushForward/PushForward_Cloud_128.h` | NK push-forward source points |

The patch bounds (π-multiples) are converted to degrees before passing to the script.

In [ ]:
gen_script = os.path.join(CODE_DIR, 'generate_pointclouds.py')

# Convert π-multiples → degrees for generate_pointclouds.py
def _deg(pi_multiple):
    return str(pi_multiple * 180.0)

print(f'Generating point clouds: NK={NK}, NK_small={NK_small} ...')
print(f'  source patch θ=[{SRC_THETA_MIN*180:.1f}°,{SRC_THETA_MAX*180:.1f}°]  '
      f'φ=[{SRC_PHI_MIN*180:.1f}°,{SRC_PHI_MAX*180:.1f}°]')
print(f'  target patch θ=[{TGT_THETA_MIN*180:.1f}°,{TGT_THETA_MAX*180:.1f}°]  '
      f'φ=[{TGT_PHI_MIN*180:.1f}°,{TGT_PHI_MAX*180:.1f}°]')

r = subprocess.run(
    [
        sys.executable, gen_script,
        str(NK), str(NK_small),
        '--src-theta-min', _deg(SRC_THETA_MIN),
        '--src-theta-max', _deg(SRC_THETA_MAX),
        '--src-phi-min',   _deg(SRC_PHI_MIN),
        '--src-phi-max',   _deg(SRC_PHI_MAX),
        '--tgt-theta-min', _deg(TGT_THETA_MIN),
        '--tgt-theta-max', _deg(TGT_THETA_MAX),
        '--tgt-phi-min',   _deg(TGT_PHI_MIN),
        '--tgt-phi-max',   _deg(TGT_PHI_MAX),
    ],
    capture_output=True, text=True
)
print(r.stdout.strip())
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError(f'generate_pointclouds.py failed (exit {r.returncode})')
print('✓ Point clouds written')

## Step 4 — Compile

Patches `main.cpp` to include the refraction benchmark header, then compiles.

The include chain automatically selects **Snell's law** (no C++ edits needed):
- `test_3D_FlatPatchRefraction_logcost_MonteCarlo.h` includes
  `Generic_3D_refractioncost_MonteCarlo.h`, which defines
  `GENERIC_3D_MONTECARLO_INCLUDED` and `Generic_3D_refractioncost_MonteCarlo`.
- `Pushforward_of_RefRegular.h` skips the logcost generic (guard fails) and
  activates the `#ifdef Generic_3D_refractioncost_MonteCarlo` Snell's law branch.

In [ ]:
import re

# Patch the #include in main.cpp to the refraction benchmark
main_cpp = os.path.join(CODE_DIR, 'main.cpp')
with open(main_cpp) as fh:
    src = fh.read()
patched = re.sub(
    r'#include\s+"Benchmarks/[^"]+"',
    f'#include "Benchmarks/{BENCHMARK}"',
    src, count=1
)
if patched != src:
    with open(main_cpp, 'w') as fh:
        fh.write(patched)
    print(f'Patched main.cpp → {BENCHMARK}')
else:
    print(f'main.cpp already set to {BENCHMARK}')

print('\nCompiling ...')
r = subprocess.run(['make', '-C', CODE_DIR], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError(f'Compilation failed (exit {r.returncode})')
print('✓ Compiled successfully')

## Step 5 — Run benchmark

The patch-bound arguments are passed as multiples of π (converted to radians inside `main()`).
The C++ `generate_patch_points()` uses these to fill `x[]` and `y[]` within each patch.

In [ ]:
import time

print(f'Running refraction benchmark (NK={NK}, κ={COST_K}) ...')
t0 = time.time()

r = subprocess.run(
    [
        os.path.join(CODE_DIR, 'main'),
        str(SRC_THETA_MIN), str(SRC_THETA_MAX),
        str(SRC_PHI_MIN),   str(SRC_PHI_MAX),
        str(TGT_THETA_MIN), str(TGT_THETA_MAX),
        str(TGT_PHI_MIN),   str(TGT_PHI_MAX),
    ],
    cwd=CODE_DIR, capture_output=True, text=True
)
elapsed = time.time() - t0

out = r.stdout
print(out[-4000:] if len(out) > 4000 else out)
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError(f'Benchmark failed (exit {r.returncode})')

print(f'✓ Completed in {elapsed:.1f}s')

## Step 6 — Find output & load helpers

In [ ]:
import glob
import numpy as np

# ── Locate the most-recently written output directory ─────────────────────────
def find_output(code_dir):
    patterns = [
        os.path.join(code_dir, 'Results_*', '**', 'Output_*'),
        os.path.join(code_dir, 'Results_*', '*'),
        os.path.join(code_dir, 'Output_*'),
    ]
    dirs = []
    for p in patterns:
        dirs += [d for d in glob.glob(p, recursive=True) if os.path.isdir(d)]
    return max(dirs, key=os.path.getmtime) if dirs else None

output_dir = find_output(CODE_DIR)
if output_dir is None:
    raise RuntimeError('No output directory found — did Step 5 succeed?')

print(f'Output directory : {output_dir}')
txts = sorted(f for f in os.listdir(output_dir) if f.endswith('.txt'))
print(f'Output files ({len(txts)}): {txts}')

# ── Data-loading helpers (used by Steps 7 & 8) ────────────────────────────────
_j = lambda name: os.path.join(output_dir, name)

def _load_vec(path):
    """Load a .txt file whose first non-empty line is a count header.
    Returns an (N, D) float array, or None if the file is missing."""
    if not os.path.exists(path):
        return None
    rows = []
    with open(path) as fh:
        fh.readline()          # skip header
        for line in fh:
            line = line.strip()
            if line:
                vals = [float(v) for v in line.split()]
                if vals:
                    rows.append(vals)
    return np.array(rows) if rows else None

def _load_pts(path):
    """Load a .txt file with no header — each line is a space-separated row."""
    if not os.path.exists(path):
        return None
    rows = []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line:
                vals = [float(v) for v in line.split()]
                if len(vals) >= 2:
                    rows.append(vals)
    return np.array(rows) if rows else None

print('✓ Helpers defined')

## Step 7 — Visualize (matplotlib, 6-panel static PNG)

Inline matplotlib visualization (not `visualize.py` — that script uses south-pole
projection for the target, which is incorrect when both patches are upper hemisphere).

Panels:
1. Source density P(x) — equirectangular grid, upper hemisphere
2. Target density Q(y) — equirectangular grid, upper hemisphere
3. Source scatter 2D — north-pole stereographic projection, coloured by P
4. Target scatter 2D — north-pole stereographic projection, coloured by Q
5. **Refractor surface 3D** — `Ref_MY.txt` coloured by radius `R_MY.txt`
6. Push-forward overlay — OT-assigned output (coloured steelblue) vs raw target (tomato)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from IPython.display import Image, display

# ── Load data ─────────────────────────────────────────────────────────────────
x_src  = _load_vec(_j('x_MY.txt'))     # (NK, 3) source directions
y_tgt  = _load_vec(_j('y_MY.txt'))     # (NK, 3) target directions
ref    = _load_vec(_j('Ref_MY.txt'))   # (NK, 3) refractor surface
p_src  = _load_vec(_j('p_MY.txt'))     # (NK, 1) source density
q_tgt  = _load_vec(_j('q_MY.txt'))     # (NK, 1) target density
g_pot  = _load_vec(_j('g_MY.txt'))     # (NK, 1) dual potential g
r_data = _load_vec(_j('R_MY.txt'))     # (NK, 1) radii
x_mesh = _load_pts(_j('X_MeshGrid.txt'))  # (180, 360) source density grid
y_mesh = _load_pts(_j('Y_MeshGrid.txt'))  # (180, 360) target density grid

for name, arr in [('x_MY', x_src), ('y_MY', y_tgt), ('Ref_MY', ref),
                  ('p_MY', p_src), ('q_MY', q_tgt), ('g_MY', g_pot)]:
    if arr is None:
        raise RuntimeError(f'{name}.txt not found — re-run Steps 5-6 first.')

p_flat = p_src.flatten()
q_flat = q_tgt.flatten()
g_flat = g_pot.flatten()
r_flat = r_data.flatten() if r_data is not None else np.ones(len(ref))

# ── North-pole stereographic projection (for upper-hemisphere points) ─────────
def stereo_north(pts):
    """Upper hemisphere → plane:  (x,y,z) → (x/(1+z), y/(1+z))"""
    denom = 1.0 + pts[:, 2]
    return pts[:, 0] / denom, pts[:, 1] / denom

# ── OT assignment for push-forward overlay ────────────────────────────────────
# j*(i) = argmin_j [ c(x_i, y_j) - g[j] ]  with c = -log(1 - κ·(x·y))
dot    = x_src @ y_tgt.T                                      # (NK, NK)
cost   = -np.log(np.clip(1.0 - COST_K * dot, 1e-12, None))   # refraction cost
assign = (cost - g_flat[np.newaxis, :]).argmin(axis=1)        # (NK,)
T_x    = y_tgt[assign]                                        # assigned targets

mask_p = p_flat > 0
mask_q = q_flat > 0

# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 11))
fig.suptitle(
    f'Refraction Benchmark — {BENCHMARK.replace(".h", "")}\n'
    f'NK={NK}  κ={COST_K}  '
    f'src θ=[{SRC_THETA_MIN:.3f}π,{SRC_THETA_MAX:.3f}π]  '
    f'tgt θ=[{TGT_THETA_MIN:.3f}π,{TGT_THETA_MAX:.3f}π]',
    fontsize=11
)

# Panel 1 — Source density (equirectangular grid)
ax1 = fig.add_subplot(2, 3, 1)
if x_mesh is not None:
    # Upper hemisphere: rows 0..89 correspond to θ ∈ [0°,90°]
    ax1.imshow(x_mesh[:90, :], origin='upper', aspect='auto',
               cmap='viridis', extent=[0, 360, 90, 0])
    ax1.set_xlabel('φ [°]'); ax1.set_ylabel('θ [°]')
ax1.set_title('Source density P(x)  (upper hemisphere)')

# Panel 2 — Target density (equirectangular grid)
ax2 = fig.add_subplot(2, 3, 2)
if y_mesh is not None:
    ax2.imshow(y_mesh[:90, :], origin='upper', aspect='auto',
               cmap='plasma', extent=[0, 360, 90, 0])
    ax2.set_xlabel('φ [°]'); ax2.set_ylabel('θ [°]')
ax2.set_title('Target density Q(y)  (upper hemisphere)')

# Panel 3 — Source scatter (north-pole stereographic)
ax3 = fig.add_subplot(2, 3, 3)
u_src, v_src = stereo_north(x_src)
sc3 = ax3.scatter(u_src[mask_p], v_src[mask_p], c=p_flat[mask_p],
                  cmap='viridis', s=10, edgecolors='none')
plt.colorbar(sc3, ax=ax3, label='P')
ax3.set_title('Source scatter  (north-pole stereo)')
ax3.set_xlabel('u'); ax3.set_ylabel('v')
ax3.set_aspect('equal')

# Panel 4 — Target scatter (north-pole stereographic)
ax4 = fig.add_subplot(2, 3, 4)
u_tgt, v_tgt = stereo_north(y_tgt)
sc4 = ax4.scatter(u_tgt[mask_q], v_tgt[mask_q], c=q_flat[mask_q],
                  cmap='plasma', s=10, edgecolors='none')
plt.colorbar(sc4, ax=ax4, label='Q')
ax4.set_title('Target scatter  (north-pole stereo)')
ax4.set_xlabel('u'); ax4.set_ylabel('v')
ax4.set_aspect('equal')

# Panel 5 — Refractor surface 3D
ax5 = fig.add_subplot(2, 3, 5, projection='3d')
sc5 = ax5.scatter(ref[mask_p, 0], ref[mask_p, 1], ref[mask_p, 2],
                  c=r_flat[mask_p], cmap='viridis', s=5, depthshade=True)
fig.colorbar(sc5, ax=ax5, label='Radius', shrink=0.6)
ax5.set_title('Refractor surface  (coloured by radius)')
ax5.set_xlabel('X'); ax5.set_ylabel('Y'); ax5.set_zlabel('Z')

# Panel 6 — Push-forward overlay
ax6 = fig.add_subplot(2, 3, 6)
u_pushed, v_pushed = stereo_north(T_x[mask_p])
ax6.scatter(u_tgt[mask_q], v_tgt[mask_q], c='tomato',
            s=6, alpha=0.5, label='Target Q(y)', edgecolors='none')
ax6.scatter(u_pushed, v_pushed, c='steelblue',
            s=6, alpha=0.5, label='Push-forward', edgecolors='none')
ax6.legend(markerscale=2)
ax6.set_title('Push-forward vs target  (north-pole stereo)')
ax6.set_xlabel('u'); ax6.set_ylabel('v')
ax6.set_aspect('equal')

plt.tight_layout(rect=[0, 0, 1, 0.93])
out_png = _j('refraction_visualization.png')
plt.savefig(out_png, dpi=120, bbox_inches='tight')
plt.close()
print(f'Saved: {out_png}')
display(Image(out_png))

## Step 8 — Interactive 3D (Plotly)

Two interactive figures saved as HTML:

**8a — Point-cloud view** (`fig_refraction_points.html`)
- Source directions Ω  (YlOrRd, coloured by P)
- Refractor surface    (Viridis, coloured by radius)
- Target directions Ω\* (Blues, coloured by Q)

**8b — Ray diagram** (`fig_refraction_rays.html`)
- Gold incoming rays  O → refractor surface
- Blue refracted rays  surface → surface + t · (OT-assigned direction)
- Red point source at origin O

Drag to rotate, scroll to zoom, click legend items to toggle.

In [ ]:
try:
    import plotly.graph_objects as go
except ImportError:
    import subprocess as _sp, sys as _sys
    _sp.run([_sys.executable, '-m', 'pip', 'install', 'plotly', '--quiet'])
    import plotly.graph_objects as go

from IPython.display import IFrame

# ── Parameters ────────────────────────────────────────────────────────────────
N_RAYS   = 150
T_OUT    = 2.0
MAX_PTS  = 4_000   # subsample for browser responsiveness

rng = np.random.default_rng(0)
active = np.where(p_flat > 0)[0]

# x_src, y_tgt, ref, r_flat, p_flat, q_flat, g_flat, assign, T_x loaded in Step 7

# ── Helper: build line segments list for ray traces ───────────────────────────
def _segs(starts, ends):
    xs, ys, zs = [], [], []
    for s, e in zip(starts, ends):
        xs += [float(s[0]), float(e[0]), None]
        ys += [float(s[1]), float(e[1]), None]
        zs += [float(s[2]), float(e[2]), None]
    return xs, ys, zs

# ════════════════════════════════════════════════════════════════════════════
# 8a — 3D point-cloud view
# ════════════════════════════════════════════════════════════════════════════
fig_pts = go.Figure()

# Subsample indices
idx_src = rng.choice(np.where(mask_p)[0], min(MAX_PTS, mask_p.sum()), replace=False)
idx_tgt = rng.choice(np.where(mask_q)[0], min(MAX_PTS, mask_q.sum()), replace=False)
idx_ref = rng.choice(len(ref),             min(MAX_PTS, len(ref)),      replace=False)

# Source Ω  (upper hemisphere, YlOrRd)
fig_pts.add_trace(go.Scatter3d(
    x=x_src[idx_src, 0], y=x_src[idx_src, 1], z=x_src[idx_src, 2],
    mode='markers',
    marker=dict(size=2.5, color=p_flat[idx_src], colorscale='YlOrRd', opacity=0.70),
    name='Source  Ω  P(x)',
    hovertemplate='x=%{x:.3f}  y=%{y:.3f}  z=%{z:.3f}<br>P=%{marker.color:.4e}<extra></extra>',
))

# Refractor surface (Viridis)
fig_pts.add_trace(go.Scatter3d(
    x=ref[idx_ref, 0], y=ref[idx_ref, 1], z=ref[idx_ref, 2],
    mode='markers',
    marker=dict(size=3, color=r_flat[idx_ref], colorscale='Viridis',
                opacity=0.85, showscale=True,
                colorbar=dict(title='Radius', x=1.02, len=0.5)),
    name='Refractor surface',
    hovertemplate='x=%{x:.3f}  y=%{y:.3f}  z=%{z:.3f}<br>R=%{marker.color:.4f}<extra></extra>',
))

# Target Ω*  (upper hemisphere, Blues)
fig_pts.add_trace(go.Scatter3d(
    x=y_tgt[idx_tgt, 0], y=y_tgt[idx_tgt, 1], z=y_tgt[idx_tgt, 2],
    mode='markers',
    marker=dict(size=2.5, color=q_flat[idx_tgt], colorscale='Blues', opacity=0.70),
    name='Target  Ω*  Q(y)',
    hovertemplate='x=%{x:.3f}  y=%{y:.3f}  z=%{z:.3f}<br>Q=%{marker.color:.4e}<extra></extra>',
))

fig_pts.update_layout(
    title=dict(
        text=(
            f'<b>Refraction Benchmark — 3D Point-Cloud View</b><br>'
            f'<sup>NK={NK}  ·  κ={COST_K}  ·  {BENCHMARK.replace(".h", "")}</sup>'
        ),
        x=0.5, xanchor='center',
    ),
    scene=dict(
        xaxis=dict(title='X', showgrid=True),
        yaxis=dict(title='Y', showgrid=True),
        zaxis=dict(title='Z', showgrid=True),
        aspectmode='data',
        camera=dict(eye=dict(x=1.4, y=1.4, z=0.8)),
    ),
    legend=dict(itemsizing='constant', x=0, y=1),
    margin=dict(l=0, r=90, t=100, b=0),
    width=960, height=720,
)

pts_html = _j('fig_refraction_points.html')
fig_pts.write_html(pts_html, include_plotlyjs='cdn')
print(f'Saved: {pts_html}')
fig_pts.show()

# ════════════════════════════════════════════════════════════════════════════
# 8b — 3D ray diagram
# ════════════════════════════════════════════════════════════════════════════

# Subsample active source rays
ray_idx = rng.choice(active, size=min(N_RAYS, len(active)), replace=False)

# Incoming: O → Ref[i]
xi, yi, zi = _segs(np.zeros((len(ray_idx), 3)), ref[ray_idx])
# Refracted: Ref[i] → Ref[i] + T_OUT * assigned_direction
xo, yo, zo = _segs(ref[ray_idx], ref[ray_idx] + T_OUT * T_x[ray_idx])

fig_rays = go.Figure()

# Refractor surface (faded background)
fig_rays.add_trace(go.Scatter3d(
    x=ref[idx_ref, 0], y=ref[idx_ref, 1], z=ref[idx_ref, 2],
    mode='markers',
    marker=dict(size=2, color=r_flat[idx_ref], colorscale='Viridis', opacity=0.35),
    name='Refractor surface', hoverinfo='skip',
))

# Incoming rays (gold)
fig_rays.add_trace(go.Scatter3d(
    x=xi, y=yi, z=zi, mode='lines',
    line=dict(color='rgba(255,160,20,0.75)', width=2),
    name=f'Incoming rays ({len(ray_idx)})', hoverinfo='skip',
))

# Refracted rays (blue)
fig_rays.add_trace(go.Scatter3d(
    x=xo, y=yo, z=zo, mode='lines',
    line=dict(color='rgba(30,120,255,0.85)', width=2),
    name=f'Refracted rays (t={T_OUT})', hoverinfo='skip',
))

# Point source O
fig_rays.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode='markers',
    marker=dict(size=6, color='red'),
    name='Point source O',
))

fig_rays.update_layout(
    title=dict(
        text=(
            f'<b>Refraction Ray Paths — {BENCHMARK.replace(".h", "")}</b><br>'
            f'<sup>NK={NK}  ·  {len(ray_idx)} rays  ·  κ={COST_K}  ·  t={T_OUT}</sup>'
        ),
        x=0.5, xanchor='center',
    ),
    scene=dict(
        xaxis=dict(title='X'), yaxis=dict(title='Y'), zaxis=dict(title='Z'),
        aspectmode='data',
        camera=dict(eye=dict(x=1.4, y=1.4, z=0.8)),
    ),
    legend=dict(itemsizing='constant', x=0, y=1),
    margin=dict(l=0, r=40, t=100, b=0),
    width=960, height=720,
)

rays_html = _j('fig_refraction_rays.html')
fig_rays.write_html(rays_html, include_plotlyjs='cdn')
print(f'Saved: {rays_html}')
fig_rays.show()

print(f'\n✓ Interactive 3D figures ready.')
print(f'  Point-cloud view : {pts_html}')
print(f'  Ray diagram      : {rays_html}')